In [153]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble  import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import warnings
warnings.filterwarnings('ignore')
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

In [82]:
df = pd.read_csv("heart.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [83]:
df.shape

(918, 12)

In [84]:
# print(df[df.Age > (df.Age.mean()) + (3 * df.Age.std())].shape)
print(df[df.RestingBP > (df.RestingBP.mean() + (3* df.RestingBP.std()))].shape)
print(df[df.Cholesterol > (df.Cholesterol.mean() + (3* df.Cholesterol.std()))].shape)
# print(df[df.FastingBS > (df.FastingBS.mean() + (3* df.FastingBS.std()))].shape)
# print(df[df.MaxHR > (df.MaxHR.mean() + (3* df.MaxHR.std()))].shape)
print(df[df.Oldpeak > (df.Oldpeak.mean() + (3* df.Oldpeak.std()))].shape)

(7, 12)
(3, 12)
(6, 12)


In [85]:
df = df[df.RestingBP <= (df.RestingBP.mean() + (3* df.RestingBP.std()))]
df = df[df.Cholesterol <= (df.Cholesterol.mean() + (3* df.Cholesterol.std()))]
df = df[df.Oldpeak <= (df.Oldpeak.mean() + (3* df.Oldpeak.std()))]
df.shape

(902, 12)

In [86]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 902 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             902 non-null    int64  
 1   Sex             902 non-null    object 
 2   ChestPainType   902 non-null    object 
 3   RestingBP       902 non-null    int64  
 4   Cholesterol     902 non-null    int64  
 5   FastingBS       902 non-null    int64  
 6   RestingECG      902 non-null    object 
 7   MaxHR           902 non-null    int64  
 8   ExerciseAngina  902 non-null    object 
 9   Oldpeak         902 non-null    float64
 10  ST_Slope        902 non-null    object 
 11  HeartDisease    902 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 91.6+ KB


In [87]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [88]:
print(df.ChestPainType.value_counts()) 
print("========================================")
print(df.Sex.value_counts())
print("========================================")
print(df.RestingECG.value_counts())
print("========================================")
print(df.ExerciseAngina.value_counts())
print("========================================")
print(df.ST_Slope.value_counts())
print("========================================")

ChestPainType
ASY    485
NAP    201
ATA    171
TA      45
Name: count, dtype: int64
Sex
M    713
F    189
Name: count, dtype: int64
RestingECG
Normal    545
LVH       182
ST        175
Name: count, dtype: int64
ExerciseAngina
N    539
Y    363
Name: count, dtype: int64
ST_Slope
Flat    452
Up      393
Down     57
Name: count, dtype: int64


In [89]:
df.ExerciseAngina.replace(
    {
        'N': 0,
        'Y': 1
    },
    inplace=True)

df.ST_Slope.replace(
    {
        'Down': 1,
        'Flat': 2,
        'Up': 3
    },
    inplace=True
)

df.RestingECG.replace(
    {
        'Normal': 1,
        'ST': 2,
        'LVH': 3
    },
    inplace=True)

df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,1,172,0,0.0,3,0
1,49,F,NAP,160,180,0,1,156,0,1.0,2,1
2,37,M,ATA,130,283,0,2,98,0,0.0,3,0
3,48,F,ASY,138,214,0,1,108,1,1.5,2,1
4,54,M,NAP,150,195,0,1,122,0,0.0,3,0


In [90]:
df = pd.get_dummies(df , drop_first=True)

In [91]:
df.head()

,Age,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA
0,40,140,289,0,1,172,0,0.0,3,0,True,True,False,False
1,49,160,180,0,1,156,0,1.0,2,1,False,False,True,False
2,37,130,283,0,2,98,0,0.0,3,0,True,True,False,False
3,48,138,214,0,1,108,1,1.5,2,1,False,False,False,False
4,54,150,195,0,1,122,0,0.0,3,0,True,False,True,False


In [94]:
bool_columns = df.select_dtypes(include='bool').columns
df[bool_columns] = df[bool_columns].astype(int)
df.head(5)

,Age,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA
0,40,140,289,0,1,172,0,0.0,3,0,1,1,0,0
1,49,160,180,0,1,156,0,1.0,2,1,0,0,1,0
2,37,130,283,0,2,98,0,0.0,3,0,1,1,0,0
3,48,138,214,0,1,108,1,1.5,2,1,0,0,0,0
4,54,150,195,0,1,122,0,0.0,3,0,1,0,1,0


In [97]:
X = df.drop('HeartDisease', axis=1)
y = df.HeartDisease

In [112]:
Scalar = StandardScaler()
X_scaled = Scalar.fit_transform(X)
print(X_scaled.shape)
print(y.shape)

(902, 13)
(902,)


In [106]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
scaled_X_train, scaled_X_test, scaled_y_train, scaled_y_test = train_test_split(X_scaled, y, test_size=0.3)

In [124]:
svm_model = SVC()
svm_model.fit(X_train, y_train)
print(svm_model.score(X_test,y_test))

0.7158671586715867


In [121]:
svm_model_sc = SVC()
svm_model_sc.fit(scaled_X_train, scaled_y_train)
svm_model_sc.score(scaled_X_test,scaled_y_test)

0.8634686346863468

In [144]:
svm_model_cvs = cross_val_score(SVC(), X, y, cv=100)
svm_model_cvs.mean()

np.float64(0.7232222222222222)

In [146]:
svm_model_cvs_sc = cross_val_score(SVC(),X_scaled, y, cv=100)
svm_model_cvs.mean()

np.float64(0.7232222222222222)

In [154]:
bag_model = BaggingClassifier(estimator=SVC(),n_estimators=10, random_state=0, max_samples=0.8)
cross_val_score(bag_model,X_scaled,y,cv=100).mean()

np.float64(0.8702222222222222)

In [155]:
bag_model = BaggingClassifier(estimator=SVC(),n_estimators=10, random_state=0, max_samples=0.8)
cross_val_score(bag_model,X,y,cv=100).mean()

np.float64(0.7165555555555556)

In [164]:
dtc_model = DecisionTreeClassifier(max_depth=100,random_state=0)
dtc_model.fit(X_train,y_train)
dtc_model.score(X_test,y_test)

0.8044280442804428

In [165]:
dtc_model = DecisionTreeClassifier(max_depth=100,random_state=0)
dtc_model.fit(X_scaled,y)
dtc_model.score(X_test,y_test)

0.7121771217712177

In [166]:
dtc_model = DecisionTreeClassifier(max_depth=100,random_state=0)
dtc_model.fit(scaled_X_train,scaled_y_train)
dtc_model.score(scaled_X_test,scaled_y_test)

0.8228782287822878

In [178]:
dtc_model = cross_val_score(DecisionTreeClassifier(max_depth=100,random_state=0), X_train,y_train, cv=5)
dtc_model.mean()

np.float64(0.7876140482439695)

In [177]:
dtc_model = cross_val_score(DecisionTreeClassifier(max_depth=100,random_state=0),scaled_X_train,scaled_y_train, cv=50)
dtc_model.mean()

np.float64(0.7808974358974359)

In [179]:
dtc_model = cross_val_score(DecisionTreeClassifier(max_depth=100,random_state=0),X_scaled,y, cv=50)
dtc_model.mean()

np.float64(0.8025730994152049)

In [180]:
bag_model = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=100,random_state=0),n_estimators=10, random_state=0, max_samples=0.8)
cross_val_score(bag_model,X,y,cv=100).mean()

np.float64(0.8368888888888891)

In [181]:
bag_model = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=100,random_state=0),n_estimators=10, random_state=0, max_samples=0.8)
cross_val_score(bag_model,X_scaled,y,cv=100).mean()

np.float64(0.8368888888888891)